# 02-Modeling：客户流失模型训练与评估

本笔记本展示完整的建模流程，与 src/main.py 对应。

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_utils import load_config
from src.viz_utils import plot_roc_curve, plot_confusion_matrix

config = load_config(project_root / "config.yaml")
config

In [ ]:
# 数据准备
from src.data_prep import clean_data, get_feature_target_split, load_data_from_hf
from src.features import build_preprocessor, encode_target, identify_column_types
from sklearn.model_selection import train_test_split

df_raw = load_data_from_hf(config["data"]["dataset-name"])
df_clean = clean_data(df_raw, config)
X, y = get_feature_target_split(df_clean, config)
y = encode_target(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

col_types = identify_column_types(X_train, config)
preprocessor = build_preprocessor(col_types, config)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print(f"训练集: {X_train_p.shape}, 测试集: {X_test_p.shape}")

In [ ]:
# 基线模型：Logistic Regression
from src.models import build_baseline_model
from src.evaluate import evaluate_model

baseline = build_baseline_model(config)
baseline.fit(X_train_p, y_train)

base_metrics = evaluate_model(baseline, X_test_p, y_test, model_name="baseline", output_dir=project_root / "outputs")
base_metrics

In [ ]:
# 主力模型：LightGBM
from src.models import build_main_model

main_model = build_main_model(config)
main_model.fit(X_train_p, y_train)

main_metrics = evaluate_model(main_model, X_test_p, y_test, model_name="lightgbm", output_dir=project_root / "outputs")
main_metrics

In [ ]:
# SHAP 可解释性分析
from src.evaluate import shap_analysis

shap_importance = shap_analysis(
    main_model, X_test_p,
    feature_names=list(X_test_p.columns),
    sample_size=500,
    output_dir=project_root / "outputs",
    model_name="lightgbm"
)

print("
Top 5 关键特征:")
print(shap_importance.head(5))